### Bidirectional Search (BS)

Bidirectional Search is an efficient graph-search method that runs **two BFS searches simultaneously**:
- One from the **start node**
- One from the **goal node**

The searches expand level-by-level until they meet in the middle.  
This reduces time complexity from **O(b^d)** (BFS) to approximately **O(b^(d/2))**, making it very powerful for large graphs.

Below, each cell implements one step of the algorithm:
- `expand_layer`: expand one BFS frontier
- `build_path`: reconstruct final path
- `bidirectional_search`: the main BS algorithm
- Finally, we test the algorithm on a sample graph


##### Step 1: expand_layer
This function expands one BFS layer from either the forward or backward frontier.
It also checks whether the two searches meet on this step.


In [1]:
from collections import deque

def expand_layer(graph, frontier, this_side, other_side):
    """Expand one BFS layer and check intersection."""
    if not frontier:
        return None

    current = frontier.popleft()

    for nb in graph[current]:
        if nb not in this_side:
            this_side[nb] = current
            frontier.append(nb)

            if nb in other_side:
                return nb

    return None


##### Step 2: build_path
Once both searches meet at a common node, this function reconstructs the full path
by joining the forward and backward parent chains.


In [2]:
def build_path(meet, visited_f, visited_b):
    """Reconstruct path from start → meet → goal."""

    path_f = []
    n = meet
    while n is not None:
        path_f.append(n)
        n = visited_f[n]
    path_f.reverse()

    path_b = []
    n = visited_b[meet]
    while n is not None:
        path_b.append(n)
        n = visited_b[n]

    return path_f + path_b


##### Step 3: bidirectional_search
This is the main BS algorithm.  
It runs two BFS frontiers (forward and backward) and uses the helper functions
from the previous cells to detect intersection and build the final path.


In [3]:
def build_reverse_graph(graph):
    r = {node: [] for node in graph}
    for node, neighbors in graph.items():
        for nb in neighbors:
            r[nb].append(node)
    return r


def bidirectional_search(graph, start, goal):
    if start == goal:
        return [start]

    reverse_graph = build_reverse_graph(graph)

    frontier_f = deque([start])
    frontier_b = deque([goal])

    visited_f = {start: None}
    visited_b = {goal: None}

    while frontier_f and frontier_b:

        meet = expand_layer(graph, frontier_f, visited_f, visited_b)
        if meet:
            return build_path(meet, visited_f, visited_b)

        meet = expand_layer(reverse_graph, frontier_b, visited_b, visited_f)
        if meet:
            return build_path(meet, visited_f, visited_b)

    return None


##### Step 4: Test the algorithm
We now test Bidirectional Search on a sample directed graph.


In [4]:
graph = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F", "G"],
    "D": ["H"],
    "E": ["I", "J"],
    "F": [],
    "G": ["K"],
    "H": [],
    "I": [],
    "J": [],
    "K": ["Goal"],
    "Goal": []
}

bidirectional_search(graph, "A", "Goal")


['A', 'C', 'G', 'K', 'Goal']